# Board Encoding Smoke Test

This notebook inspects the McChess single-board encoder, including the en-passant target plane. It is for quick local sanity checks, not reportable experiments.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
import chess

from mcchess.board import PLANE_NAMES, encode_board, square_to_tensor_coords

In [ ]:
board = chess.Board()
tensor = encode_board(board)

assert tensor.shape == (18, 8, 8)
assert tensor.dtype.name == "float32"
assert PLANE_NAMES[-1] == "en_passant_target"

tensor.shape, tensor.dtype, PLANE_NAMES

In [ ]:
[(name, int(tensor[index].sum())) for index, name in enumerate(PLANE_NAMES)]

In [ ]:
{
    "a8": square_to_tensor_coords(chess.A8),
    "h8": square_to_tensor_coords(chess.H8),
    "a1": square_to_tensor_coords(chess.A1),
    "h1": square_to_tensor_coords(chess.H1),
}

In [ ]:
board.push(chess.Move.from_uci("e2e4"))
after_e4 = encode_board(board)

{
    "fen": board.fen(),
    "white_to_move_plane_sum": float(after_e4[PLANE_NAMES.index("white_to_move")].sum()),
    "white_pawns": int(after_e4[PLANE_NAMES.index("white_pawn")].sum()),
    "en_passant_target_sum": float(after_e4[PLANE_NAMES.index("en_passant_target")].sum()),
}

In [ ]:
ep_board = chess.Board()
for uci in ("e2e4", "a7a6", "e4e5", "d7d5"):
    ep_board.push(chess.Move.from_uci(uci))

ep_tensor = encode_board(ep_board)
ep_plane = ep_tensor[PLANE_NAMES.index("en_passant_target")]
d6_row, d6_col = square_to_tensor_coords(chess.D6)

assert ep_board.has_legal_en_passant()
assert ep_board.ep_square == chess.D6
assert ep_plane[d6_row, d6_col] == 1.0
assert int(ep_plane.sum()) == 1

{
    "fen": ep_board.fen(),
    "ep_square": chess.square_name(ep_board.ep_square),
    "encoded_square": "d6",
    "en_passant_target_sum": float(ep_plane.sum()),
}

In [ ]:
stale_ep_board = chess.Board(
    "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"
)
stale_ep_plane = encode_board(stale_ep_board)[PLANE_NAMES.index("en_passant_target")]

assert stale_ep_board.ep_square == chess.E3
assert not stale_ep_board.has_legal_en_passant()
assert int(stale_ep_plane.sum()) == 0

{
    "fen": stale_ep_board.fen(),
    "fen_ep_square": chess.square_name(stale_ep_board.ep_square),
    "legal_en_passant": stale_ep_board.has_legal_en_passant(),
    "encoded_sum": float(stale_ep_plane.sum()),
}